In [4]:
%pip install youtube-transcript-api

from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

def extract_video_id(url: str) -> str:
    """
    Extract the YouTube video ID from a URL.
    Raises ValueError if no 'v' parameter is found.
    """
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')

    if not video_ids:
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

if __name__ == "__main__":
    url = "https://www.youtube.com/watch?v=svPXLcrzU-w"

    video_id = extract_video_id(url)

    api = YouTubeTranscriptApi()
    fetched = api.fetch(video_id, languages=['en']) # returns a FetchedTranscript

    text = "\n".join(snippet.text for snippet in fetched)

print(text)

Note: you may need to restart the kernel to use updated packages.
It's going to be lightning session,
jam-packed, action-packed. We will have
no time to cover everything. The
intention of this day is not to solve
all the problems of a medic education.
If you've kind of figured that out,
we're going to give you a preview of
some of it, seeing the material you can
use and reuse, but the point is to
follow up and we'll have a whole hour
reception at the end of the session
where we encourage you to go right up to
the speakers and panelists and discuss
further. So, without further ado, let's
start diving in with our first slang
demo from Dr. Prius. would you like to
um take it away from our emergency
medicine talking about many different
educational applications? Please take it
away. Thank you.
[Applause]
>> Thank you for having me. Uh my name is
Dr. Brookshitus. I'm an emergency
physician and a medical educator in the
department of emergency medicine. I also
co-direct uh the human experien

In [18]:
def chunk_transcript(text: str, max_words: int = 1500) -> list:
    """
    Splits a transcript into clean paragraphs/sections 
    without severing mid-sentence thoughts.
    """
    words = text.split()
    chunks = []
    current_chunk = []
    
    for word in words:
        current_chunk.append(word)
        # Create a boundary once word limit is hit and sentence naturally ends
        if len(current_chunk) >= max_words and word.endswith(('.', '!', '?')):
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks

chunks = chunk_transcript(text, 100)
print(chunks[1])

Prius. would you like to um take it away from our emergency medicine talking about many different educational applications? Please take it away. Thank you. [Applause] >> Thank you for having me. Uh my name is Dr. Brookshitus. I'm an emergency physician and a medical educator in the department of emergency medicine. I also co-direct uh the human experience and advancement lab. And our focus is really on using AI and informatics tools to better understand how humans perform within clinical environments and how they develop. Let's start with this. I think this is what we think of most of the time when we think of medical education like a nice controlled classroom environment.


In [20]:
MAP_SYSTEM_PROMPT = """
You are an advanced text analyst. Summarize the following section of a video transcript.
Extract key concepts, decisions made, and technical tools highlighted. 
Keep your summary factual and strictly derived from the provided text.
"""

REDUCE_SYSTEM_PROMPT = """
You are a technical editor. Combine the following chronological summaries from a video presentation into a single comprehensive synthesis.
Format your final response cleanly using Markdown with these exact sections:
## Executive Summary
## Main Themes & Technical Tools Highlighted
## Key Takeaways
"""

In [41]:
import time
import os
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError  # Added ClientError to catch 429
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

MAP_SYSTEM_PROMPT = "You are an expert editor. Summarize this section of a video transcript cleanly and factually."
REDUCE_SYSTEM_PROMPT =  """
    You are a master technical editor. Analyze the provided video transcript and generate a comprehensive summary.
    Format your final response cleanly using Markdown with these exact sections:
    ## Executive Summary
    ## Main Themes & Technical Tools Highlighted
    ## Key Takeaways
    """

def generate_summary(transcript_text: str) -> str:
    chunks = chunk_transcript(transcript_text, max_words=3000)
    intermediate_summaries = []

    print(f"Processing {len(chunks)} transcript sections...")

    # 4. Map Stage with Rate Limit & Traffic Management
    for i, chunk in enumerate(chunks):
        retries = 5
        
        while retries > 0:
            try:
                response = client.models.generate_content(
                    model='gemini-2.5-flash',
                    contents=chunk,
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        system_instruction=MAP_SYSTEM_PROMPT
                    )
                )
                intermediate_summaries.append(response.text)
                
                # --- RATE LIMIT SAFEGUARD ---
                # Free tier allows 5 requests per minute. 
                # Pausing for 12 seconds between successful chunks guarantees we stay under the limit.
                if i < len(chunks) - 1:
                    print(f"✅ Chunk {i+1}/{len(chunks)} processed successfully. Pacing API requests...")
                    time.sleep(12) 
                
                break  # Success! Break the retry loop and move to next chunk
                
            except ClientError as e:
                # Catching the 429 Rate Limit error explicitly
                if e.code == 429:
                    retries -= 1
                    print(f"⏳ Rate limit hit (429) on chunk {i+1}. Sleeping for 45 seconds to reset quota... ({retries} retries left)")
                    time.sleep(45)  # Sleep long enough to reset the 1-minute window
                else:
                    raise e  # If it's a different client error (e.g., bad format), raise it
                    
            except ServerError as e:
                # Catching the 503 Traffic spike error
                if e.code == 503:
                    retries -= 1
                    print(f"⚠️ Server busy (503) on chunk {i+1}. Retrying in 10 seconds...")
                    time.sleep(10)
                else:
                    raise e
        else:
            raise RuntimeError(f"❌ Failed to process chunk {i+1} after multiple cooling-off periods.")

    # 5. Reduce Stage
    print("\nSynthesizing final summary...")
    # Optional short pause to ensure the final Reduce call has fresh quota space
    time.sleep(5)
    
    combined_intermediates = "\n\n--- Next Section Summary ---\n\n".join(intermediate_summaries)
    
    final_response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=combined_intermediates,
        config=types.GenerateContentConfig(
            temperature=0.3,
            system_instruction=REDUCE_SYSTEM_PROMPT
        )
    )
    
    return final_response.text

# --- Test Execution ---
summary_output = generate_summary(text)
print(summary_output)

Processing 4 transcript sections...
✅ Chunk 1/4 processed successfully. Pacing API requests...
✅ Chunk 2/4 processed successfully. Pacing API requests...
✅ Chunk 3/4 processed successfully. Pacing API requests...

Synthesizing final summary...
## Executive Summary

This session showcased the transformative potential of AI in medical education, moving beyond traditional knowledge assessment to address complex aspects of clinical performance, reasoning, and behavior. Speakers, including educators, a medical student, and platform developers, presented a diverse range of AI applications. These ranged from personal study tools that streamline information processing and material creation, to sophisticated institutional platforms designed for scalable patient simulations, diagnostic reasoning practice, and the cultivation of critical thinking. A central theme was AI's ability to illuminate unseen patterns, map learning trajectories, measure new aspects of performance, and provide targeted, ac

In [36]:
import os
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# 1. Initialize with your fresh API key
user_secrets = UserSecretsClient()
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

def generate_single_call_summary(transcript_text: str) -> str:
    print("Sending entire transcript to Gemini in a single request...")
    
    SYSTEM_PROMPT = """
    You are a master technical editor. Analyze the provided video transcript and generate a comprehensive summary.
    Format your final response cleanly using Markdown with these exact sections:
    ## Executive Summary
    ## Main Themes & Technical Tools Highlighted
    ## Key Takeaways
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=transcript_text,
            config=types.GenerateContentConfig(
                temperature=0.3,
                system_instruction=SYSTEM_PROMPT
            )
        )
        return response.text
    except ClientError as e:
        if e.code == 429:
            print("Rate limit hit. Wait 60 seconds and run this cell again.")
        raise e

# Run it cleanly in one shot
summary_output = generate_single_call_summary(text)
print(summary_output)

Sending entire transcript to Gemini in a single request...
## Executive Summary

The session provided a rapid-fire overview of how Artificial Intelligence (AI) is transforming medical education, moving beyond traditional knowledge acquisition to enhance clinical reasoning, performance, and efficiency in real-world settings. Speakers from diverse backgrounds—emergency medicine, medical students, commercial developers, and learning scientists—showcased innovative AI applications. Key themes included leveraging AI for personalized learning, performance analytics, scalable clinical simulations, real-time clinical support, and fostering critical thinking habits. The overarching message emphasized AI's role in augmenting human educators and learners, providing tools to tackle complex educational challenges and prepare future healthcare professionals for dynamic clinical environments.

## Main Themes & Technical Tools Highlighted

The presentations covered a wide spectrum of AI applications i